In [ ]:
!pip install mlflow

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter, defaultdict

import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

In [ ]:
TRACKING_URI = ""
EXPERIMENT = "rubert_finetuning"
RUN_NAME = "socials df triplet loss rubertBase"

#MLFLOW_PREPROCESS_RUN_ID = "406f39f4f7844e10b58b1b508d6f2bc7"
MLFLOW_PREPROCESS_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_RUN_ID = "e48ec02695424bedaab895917a6b19f1"
MLFLOW_DF_ARTIFACT_PATH = "data/multigenres_df.csv"

RANDOM_STATE = 67

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.end_run()
mlflow.set_experiment(EXPERIMENT)
mlflow.start_run(run_name=RUN_NAME)

mlflow.set_tag("upstream.preprocess_run_id", MLFLOW_PREPROCESS_RUN_ID)
mlflow.log_param("random_state", RANDOM_STATE)

client = MlflowClient()

In [ ]:
local_path = client.download_artifacts(run_id=MLFLOW_DF_ARTIFACT_RUN_ID, path=MLFLOW_DF_ARTIFACT_PATH)
#local_path = "balanced_lit.csv"
import pandas as pd
data = pd.read_csv(local_path)
data = data[data['source_type'] == 'socials']
data.head()

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

In [ ]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"
#MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nДевайс: {DEVICE}")
mlflow.log_param("device", DEVICE)

In [ ]:
class TripletDataset(Dataset):

    def __init__(self, corpus: dict, tokenizer, max_len: int = 128, size: int = 500):
        self.corpus = corpus
        self.authors = list(corpus.keys())
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.size = size

    def __len__(self):
        return self.size

    def _tokenize(self, text: str) -> dict:
        return self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        anchor_author = random.choice(self.authors)
        chunks = self.corpus[anchor_author]

        if len(chunks) < 2:
            raise ValueError(f"У автора {anchor_author} < 2 фрагментов")

        anchor_text, positive_text = random.sample(chunks, 2)

        other_authors = [a for a in self.authors if a != anchor_author]
        neg_author = random.choice(other_authors)
        negative_text = random.choice(self.corpus[neg_author])

        anchor_enc   = self._tokenize(anchor_text)
        positive_enc = self._tokenize(positive_text)
        negative_enc = self._tokenize(negative_text)

        return {
            "anchor_ids":   anchor_enc["input_ids"].squeeze(0),
            "anchor_mask":  anchor_enc["attention_mask"].squeeze(0),
            "positive_ids": positive_enc["input_ids"].squeeze(0),
            "positive_mask":positive_enc["attention_mask"].squeeze(0),
            "negative_ids": negative_enc["input_ids"].squeeze(0),
            "negative_mask":negative_enc["attention_mask"].squeeze(0),
        }

In [ ]:
class SiameseBERT(nn.Module):

    def __init__(self, model_name: str, embedding_dim):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden_size = self.bert.config.hidden_size

        self.projection = nn.Sequential(
            nn.Linear(bert_hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, embedding_dim)
        )

    def encode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        cls_output = outputs.last_hidden_state[:, 0, :]

        projected = self.projection(cls_output)

        normalized = F.normalize(projected, p=2, dim=-1)

        return normalized

    def forward(self, batch: dict) -> tuple:
        emb_anchor   = self.encode(batch["anchor_ids"],   batch["anchor_mask"])
        emb_positive = self.encode(batch["positive_ids"], batch["positive_mask"])
        emb_negative = self.encode(batch["negative_ids"], batch["negative_mask"])

        return emb_anchor, emb_positive, emb_negative

In [ ]:
def compute_triplet_accuracy(emb_a, emb_p, emb_n) -> float:
    d_pos = F.pairwise_distance(emb_a, emb_p)
    d_neg = F.pairwise_distance(emb_a, emb_n)
    correct = (d_pos < d_neg).float().mean().item()
    return correct

In [ ]:
def freeze_lower_layers(model, freeze_until_layer: int = 8):
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    for i, layer in enumerate(model.bert.encoder.layer):
        if i < freeze_until_layer:
            for param in layer.parameters():
                param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())

In [ ]:
def train(
    model: SiameseBERT,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 5,
    lr: float = 2e-5,
    margin: float = 0.5,
    device: str = "cpu"
):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    criterion = nn.TripletMarginLoss(margin=margin, p=2)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        train_loss, train_acc = 0.0, 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()

            emb_a, emb_p, emb_n = model(batch)

            loss = criterion(emb_a, emb_p, emb_n)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()
            train_acc  += compute_triplet_accuracy(emb_a, emb_p, emb_n)

        train_loss /= len(train_loader)
        train_acc  /= len(train_loader)

        model.eval()
        val_loss, val_acc = 0.0, 0.0

        mlflow.log_metric('train_loss', train_loss, step=epoch)
        mlflow.log_metric('train_acc', train_acc, step=epoch)

        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                emb_a, emb_p, emb_n = model(batch)
                loss = criterion(emb_a, emb_p, emb_n)
                val_loss += loss.item()
                val_acc  += compute_triplet_accuracy(emb_a, emb_p, emb_n)

        val_loss /= len(val_loader)
        val_acc  /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2%} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2%}")

        mlflow.log_metric('val_loss', val_loss, step=epoch)
        mlflow.log_metric('val_acc', val_loss, step=epoch)

    return history

In [ ]:
def df_to_corpus(df: pd.DataFrame,
                 text_col:   str = 'text',
                 author_col: str = 'author') -> dict:
    return {
        author: group[text_col].tolist()
        for author, group in df.groupby(author_col)
    }

In [ ]:
def df_to_source_corpus(df: pd.DataFrame,
                 author_col: str = 'author',
                 source_col: str = 'source_type') -> dict:
    return {
        author: group[source_col].tolist()[0]
        for author, group in df.groupby(author_col)
    }

In [ ]:
FREEZE_LAYERS = 8
BATCH_SIZE = 16
EMBEDDING_DIM = 256
MARGIN = 0.2
LEARNING_RATE = 2e-5
EPOCHS = 5
TEST_SPLIT_SIZE = 0.2

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(data, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_STATE)

train_corpus = df_to_corpus(train_df)
val_corpus   = df_to_corpus(test_df)

# Проверка — убеждаемся что у всех авторов минимум 2 фрагмента
short = {a: len(t) for a, t in train_corpus.items() if len(t) < 2}
if short:
    raise ValueError(f"Для triplet loss на автора нужно >= 2 текстов:\n {short}")

In [ ]:
train_ds = TripletDataset(train_corpus, tokenizer, max_len=512, size=5000)
val_ds   = TripletDataset(val_corpus,   tokenizer, max_len=512, size=1000)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = SiameseBERT(model_name=MODEL_NAME, embedding_dim=EMBEDDING_DIM)

freeze_lower_layers(model, freeze_until_layer=FREEZE_LAYERS)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
total_params

In [ ]:
mlflow.log_params(
    {
        'freeze_layers': FREEZE_LAYERS,
        'batch_size': BATCH_SIZE,
        'test_split_size': TEST_SPLIT_SIZE,
        'epochs': EPOCHS,
        'tokenzier': MODEL_NAME,
        'device': DEVICE,
        'learning_rate': LEARNING_RATE,
        'margin': MARGIN,
        'embedding_dim': EMBEDDING_DIM,
        'random_state': RANDOM_STATE,
        'total_params': total_params,
        'epoch_batches_count': 5000
    }
)

mlflow_log_dataset = mlflow.data.from_pandas(data, name="socials_only_from_multigenre")
mlflow.log_input(mlflow_log_dataset)

In [ ]:
history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        margin=MARGIN,
        device=DEVICE
    )
history

In [ ]:
# чтобы не считать эмбеддинги заново каждый раз + поиск
class EmbeddingStore:

    def __init__(self):
        self.embeddings = []
        self.authors    = []
        self.texts      = []
        self.sources    = []

    def add(self, author: str, embedding: np.ndarray, text: str = "", source="UNK"):
        self.embeddings.append(embedding)
        self.authors.append(author)
        self.texts.append(text)
        self.sources.append(source)

    def search(self, query_embedding: np.ndarray, k: int = 5) -> list[dict]:
        # k-nearest-neighbours по косинусному сходству.
        if not self.embeddings:
            return []

        matrix = np.stack(self.embeddings)
        query  = query_embedding.reshape(1, -1)

        sims = cosine_similarity(query, matrix)[0]

        top_k_idx = np.argsort(sims)[::-1][:k]

        results = []
        for idx in top_k_idx:
            results.append({
                "author":     self.authors[idx],
                "similarity": float(sims[idx]),
                "text":       self.texts[idx][:60] + "..."
            })

        return results

    def get_matrix(self) -> np.ndarray:
        return np.stack(self.embeddings)

    def get_authors(self) -> list:
        return self.authors

In [ ]:
def build_embedding_store(
    model: SiameseBERT,
    corpus: dict,
    source_corpus: dict,
    device: str = "cpu"
) -> EmbeddingStore:
    store = EmbeddingStore()
    model.eval()

    with torch.no_grad():
        for author, chunks in corpus.items():
            for chunk in chunks:
                enc = tokenizer(
                    chunk,
                    max_length=128,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                emb = model.encode(
                    enc["input_ids"].to(device),
                    enc["attention_mask"].to(device)
                )
                store.add(author, emb.cpu().numpy().squeeze(), text=chunk, source=source_corpus[author])
    return store

In [ ]:
def predict_author(
    text: str,
    model: SiameseBERT,
    store: EmbeddingStore,
    k: int = 5,
    threshold: float = 0.5,
    device: str = "cpu"
) -> dict:
    """
    считает эмбеддинг, смотрит ближайших соседей, выбирает самого часто встречающегося автора
    """
    model.eval()
    with torch.no_grad():
        enc = tokenizer(
            text,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        query_emb = model.encode(
            enc["input_ids"].to(device),
            enc["attention_mask"].to(device)
        ).cpu().numpy().squeeze()

    neighbors = store.search(query_emb, k=k)

    votes = Counter(n["author"] for n in neighbors)
    avg_sim = np.mean([n["similarity"] for n in neighbors])
    top_sim = neighbors[0]["similarity"] if neighbors else 0

    predicted = votes.most_common(1)[0][0]

    # если самый ближайший слишком далеко - не понятно, кто это
    if top_sim < threshold:
        predicted = "Неизвестный автор"

    return {
        "predicted":  predicted,
        "confidence": top_sim,
        "avg_sim":    avg_sim,
        "votes":      dict(votes),
        "neighbors":  neighbors
    }

In [ ]:
# для проверки того, как много верных эмбеддингов(с тем же автором) рядом с текстом
def evaluate_map_at_k_fast(
    model: SiameseBERT,
    corpus: dict,
    store: EmbeddingStore,
    k: int = 5,
    device: str = "cpu",
    batch_size: int = 64
) -> float:
    model.eval()

    all_chunks       = []
    all_true_authors = []

    for author, chunks in corpus.items():
        for chunk in chunks:
            all_chunks.append(chunk)
            all_true_authors.append(author)

    all_embeddings = []

    # батчами быстрее считать
    for i in range(0, len(all_chunks), batch_size):
        batch_texts = all_chunks[i : i + batch_size]

        enc = tokenizer(
            batch_texts,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            emb = model.encode(
                enc["input_ids"].to(device),
                enc["attention_mask"].to(device)
            )

        all_embeddings.append(emb.cpu().numpy())

    query_matrix = np.vstack(all_embeddings)

    store_matrix  = store.get_matrix()
    store_authors = store.get_authors()

    sim_matrix = query_matrix @ store_matrix.T

    ap_scores = []

    for i in range(len(all_chunks)):
        true_author = all_true_authors[i]
        chunk       = all_chunks[i]

        sims    = sim_matrix[i]
        top_idx = np.argsort(sims)[::-1][: k + 1]

        correct    = 0
        precisions = []

        for rank, j in enumerate(top_idx):
            if len(precisions) == k:
                break

            # скип проверяемого текста
            if store.texts[j][:60] == chunk[:60]:
                continue

            if store_authors[j] == true_author:
                correct += 1
                precisions.append(correct / (rank + 1))

        ap = np.mean(precisions) if precisions else 0.0
        ap_scores.append(ap)

    return float(np.mean(ap_scores))

In [ ]:
full_corpus = df_to_corpus(data)

In [ ]:
data['source_type'] = 'ruslit'

In [ ]:
source_corpus = df_to_source_corpus(data, 'author', 'source_type')

In [ ]:
store = build_embedding_store(model, full_corpus, source_corpus, device=DEVICE)

In [ ]:
for k in [1, 3, 10]:
  map_at = evaluate_map_at_k_fast(model, full_corpus, store, k=k, device=DEVICE)
  print(f"MAP@{k} (обученные авторы): {map_at:.4f}")
  mlflow.log_metric(f'map_at_{k}', map_at)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA


def get_embeddings_for_viz(store: EmbeddingStore,
                            max_per_author: int = 50):
    """
    Берём по max_per_author эмбеддингов от каждого автора.
    Ограничиваем чтобы график не был перегружен.
    """
    from collections import defaultdict

    author_indices = defaultdict(list)
    for i, author in enumerate(store.authors):
        author_indices[author].append(i)

    selected_indices = []
    selected_authors = []

    for author, indices in author_indices.items():
        chosen = indices[:max_per_author]
        selected_indices.extend(chosen)
        selected_authors.extend([author] * len(chosen))

    matrix = np.stack(store.embeddings)[selected_indices]
    return matrix, selected_authors


def reduce_2d(matrix: np.ndarray, method: str = 'tsne') -> np.ndarray:
    """
    Понижение размерности до 2D.

    Сначала PCA до 50 измерений — ускоряет t-SNE без потери структуры.
    Затем t-SNE или UMAP до 2D.

    method: 'tsne' или 'umap'
    """
    # PCA — предварительное сжатие
    n_pca = min(50, matrix.shape[1], matrix.shape[0] - 1)
    pca   = PCA(n_components=n_pca, random_state=42)
    reduced = pca.fit_transform(matrix)

    if method == 'tsne':
        tsne = TSNE(
            n_components=2,
            perplexity=min(30, len(matrix) // 5),
            random_state=42,
            n_iter=1000,
            metric='cosine'   # эмбеддинги нормализованы — косинус уместен
        )
        return tsne.fit_transform(reduced)

    elif method == 'umap':
        import umap
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=15,
            min_dist=0.1,
            metric='cosine',
            random_state=42
        )
        return reducer.fit_transform(reduced)


def reduce_3d(matrix: np.ndarray) -> np.ndarray:
    """Понижение до 3D через UMAP — t-SNE в 3D работает хуже."""
    import umap
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )
    n_pca   = min(50, matrix.shape[1], matrix.shape[0] - 1)
    reduced = PCA(n_components=n_pca, random_state=42).fit_transform(matrix)
    return reducer.fit_transform(reduced)


# ── 2D график ──────────────────────────────────────────────────────

def plot_2d(store: EmbeddingStore,
            method: str = 'tsne',
            max_per_author: int = 50,
            figsize: tuple = (14, 10)):

    matrix, authors = get_embeddings_for_viz(store, max_per_author)
    coords          = reduce_2d(matrix, method=method)
    unique_authors  = sorted(set(authors))

    # Цвета — tab20 для до 20 авторов, иначе hsv
    cmap   = plt.cm.get_cmap('tab20' if len(unique_authors) <= 20 else 'hsv',
                              len(unique_authors))
    colors = {a: cmap(i) for i, a in enumerate(unique_authors)}

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')

    for author in unique_authors:
        mask = [a == author for a in authors]
        xs   = coords[mask, 0]
        ys   = coords[mask, 1]
        ax.scatter(xs, ys,
                   c=[colors[author]],
                   label=author,
                   alpha=0.7,
                   s=25,
                   edgecolors='none')

    ax.set_title(f'Эмбеддинги авторов ({method.upper()})',
                 color='white', fontsize=14, pad=15)
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_color('#333355')

    legend = ax.legend(
        bbox_to_anchor=(1.02, 1), loc='upper left',
        fontsize=7, ncol=max(1, len(unique_authors) // 25),
        facecolor='#1a1a2e', edgecolor='#333355', labelcolor='white'
    )

    plt.tight_layout()
    plt.savefig(f'embeddings_2d_{method}.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    print(f"Сохранено: embeddings_2d_{method}.png")

In [ ]:
SOURCE_COLORS = {
    'ruslit':   '#3266ad',
    'nplus1':  '#1D9E75',
    'social': '#D85A30',
    'UNK':      '#888780'
}

SOURCE_LABELS = {
    'ruslit':   'Литература',
    'nplus1':  'Научпоп',
    'social': 'Соцсети',
    'UNK':      'Неизвестно'
}


def get_embeddings_for_viz1(store: EmbeddingStore,
                            max_per_author: int = 50):
    from collections import defaultdict
    author_indices = defaultdict(list)
    for i, author in enumerate(store.authors):
        author_indices[author].append(i)

    selected = []
    for author, indices in author_indices.items():
        selected.extend(indices[:max_per_author])

    matrix  = np.stack(store.embeddings)[selected]
    authors = [store.authors[i] for i in selected]
    # sources берём если поле есть, иначе заполняем 'unknown'
    sources = [store.sources[i] if hasattr(store, 'sources') and i < len(store.sources)
               else 'unknown' for i in selected]

    return matrix, authors, sources


def plot_2d_t(store: EmbeddingStore,
            method:         str   = 'tsne',
            color_by:       str   = 'author',   # 'author' | 'source' | 'both'
            max_per_author: int   = 50,
            figsize:        tuple = (14, 10)):
    """
    color_by='author'  — каждый автор своим цветом (как раньше)
    color_by='source'  — каждый источник своим цветом
    color_by='both'    — цвет = автор, контур точки = источник
    """
    matrix, authors, sources = get_embeddings_for_viz1(store, max_per_author)
    coords         = reduce_2d(matrix, method=method)
    unique_authors = sorted(set(authors))

    cmap   = plt.cm.get_cmap('tab20' if len(unique_authors) <= 20 else 'hsv',
                              len(unique_authors))
    author_colors = {a: cmap(i) for i, a in enumerate(unique_authors)}

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')

    if color_by == 'author':
        for author in unique_authors:
            mask = np.array([a == author for a in authors])
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=[author_colors[author]],
                       label=author, alpha=0.7, s=25, edgecolors='none')

        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
                  fontsize=7, ncol=max(1, len(unique_authors) // 25),
                  facecolor='#1a1a2e', edgecolor='#333355', labelcolor='white')

    elif color_by == 'source':
        unique_sources = sorted(set(sources))
        for src in unique_sources:
            mask = np.array([s == src for s in sources])
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=SOURCE_COLORS.get(src, '#888780'),
                       label=SOURCE_LABELS.get(src, src),
                       alpha=0.7, s=25, edgecolors='none')

        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
                  fontsize=10, facecolor='#1a1a2e',
                  edgecolor='#333355', labelcolor='white')

    elif color_by == 'both':
        # Заливка = цвет автора, контур = цвет источника
        for i, (author, src) in enumerate(zip(authors, sources)):
            ax.scatter(coords[i, 0], coords[i, 1],
                       c=[author_colors[author]],
                       edgecolors=SOURCE_COLORS.get(src, '#888780'),
                       linewidths=0.8, s=35, alpha=0.8)

        # Две отдельные легенды
        import matplotlib.patches as mpatches
        author_handles = [
            mpatches.Patch(color=author_colors[a], label=a)
            for a in unique_authors
        ]
        source_handles = [
            mpatches.Patch(facecolor='#0f0f1a',
                           edgecolor=SOURCE_COLORS.get(s, '#888780'),
                           linewidth=1.5,
                           label=SOURCE_LABELS.get(s, s))
            for s in sorted(set(sources))
        ]
        l1 = ax.legend(handles=author_handles, bbox_to_anchor=(1.02, 1),
                       loc='upper left', fontsize=7,
                       ncol=max(1, len(unique_authors) // 25),
                       facecolor='#1a1a2e', edgecolor='#333355',
                       labelcolor='white', title='Авторы')
        l1.get_title().set_color('white')
        ax.add_artist(l1)
        ax.legend(handles=source_handles, bbox_to_anchor=(1.02, 0),
                  loc='lower left', fontsize=9,
                  facecolor='#1a1a2e', edgecolor='#333355',
                  labelcolor='white', title='Источник (контур)')

    ax.set_title(f'Эмбеддинги авторов ({method.upper()})  |  цвет: {color_by}',
                 color='white', fontsize=14, pad=15)
    ax.tick_params(colors='gray')
    for spine in ax.spines.values():
        spine.set_color('#333355')

    plt.tight_layout()
    plt.savefig(f'embeddings_2d_{method}_{color_by}.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()

In [ ]:
plot_2d(store, method='umap', max_per_author=50)

In [ ]:
plot_2d_t(store, method='umap', color_by='source')

In [ ]:
mlflow.log_artifact('embeddings_2d_umap.png', 'plots/embeddings_2d_umap.png')
mlflow.log_artifact('embeddings_2d_umap_source.png', 'plots/embeddings_2d_umap_source.png')

In [ ]:
mlflow.end_run()